## Exercise 4: Data Warehouse Querying and Basic Geospatial Operations

Skills: 
* Query data warehouse table
* Use dictionary to map values

References: 
* https://cityoflosangeles.github.io/best-practices/spatial-analysis-basics.html
* https://cityoflosangeles.github.io/best-practices/spatial-analysis-intro.html
* https://cityoflosangeles.github.io/best-practices/spatial-analysis-intermediate.html
* https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_examples/warehouse_tutorial.html
* https://docs.calitp.org/data-infra/analytics_examples/new_tutorial.html

In [1]:
import geopandas as gpd
import pandas as pd
import os

#os.environ["CALITP_BQ_MAX_BYTES"] = str(100_000_000_000)
pd.set_option("display.max_rows", 20)

from calitp.tables import tbl
from calitp import query_sql
from siuba import *
import shared_utils

/opt/conda/lib/python3.9/site-packages/geopandas/_compat.py:111: UserWarning: The Shapely GEOS version (3.9.1-CAPI-1.14.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(


## Query a table, turn it into a gdf
For two operators, `ITP_ID == 246` (Caltrain), `ITP_ID == 279` (BART) and `ITP_ID == 343` (Merced the Bus), you will query the warehouse table.

* Query `views.gtfs_schedule_dim_stops`
* Filter to the ITP_ID of interest
* Select these columns: `calitp_itp_id`, `stop_id`, `stop_lat`, `stop_lon`, `stop_name`
* Return as a dataframe using `collect()`
* Turn the point data into geometry with `geopandas`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.points_from_xy.html)


In [2]:
ITP_ID = [246, 279, 343]
stops = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select(_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id.isin(ITP_ID))
)

/opt/conda/lib/python3.9/site-packages/siuba/sql/utils.py:52: SAWarning: Dialect bigquery:bigquery will not make use of SQL compilation caching as it does not set the 'supports_statement_cache' attribute to ``True``.  This can have significant performance implications including some performance degradations in comparison to prior SQLAlchemy versions.  Dialect maintainers should seek to set this attribute to True after appropriate development and testing for SQLAlchemy 1.4 caching support.   Alternatively, this attribute may be set to False which will disable this warning. (Background on this error at: https://sqlalche.me/e/14/cprf)


## Use a dictionary to map values

* Create a new column called `operator` where `itp_id` is associated with its operator name.
* First, write a function to do it.
* Then, use a dictionary to do it (create new column called `agency`).
* Double check that `operator` and `agency` show the same values. Use `assert` to check.
    * `assert df.operator == df.agency`
* Hint: https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html

In [3]:
#previewing
stops.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name
396506,246,22nd_street,37.756972,-122.392492,22nd Street Station
396507,246,22nd_street,37.756972,-122.392492,22nd Street


In [4]:
stops.dtypes

itp_id         int64
stop_id       object
stop_lat     float64
stop_lon     float64
stop_name     object
dtype: object

#### Function Method

In [5]:
def operator_names(x):
    if x.itp_id == 246:
        return "Caltrain"
    elif x.itp_id == 279:
        return "BART"
    else:
        return "Merced the Bus"

In [6]:
stops["operator"] = stops.apply(lambda x: operator_names(x), axis=1) 

#### Dictionary Method

In [7]:
agency_names = {279: 'BART', 246:'Caltrain', 343: 'Merced the Bus'}

In [8]:
stops["agency"] = stops.itp_id.map(agency_names)

In [9]:
stops.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,operator,agency
396506,246,22nd_street,37.756972,-122.392492,22nd Street Station,Caltrain,Caltrain
396507,246,22nd_street,37.756972,-122.392492,22nd Street,Caltrain,Caltrain


#### Checking that they are the same 
<b> QUESTION:</b>When I use assert it says "The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all()."

In [10]:
#assert stops.operator == stops.agency

In [11]:
#checking if the 2 columns match  another way 
stops['operator'].equals(stops['agency']) 

True

## Turn lat/lon into point geometry

There is a [function in shared_utils](https://github.com/cal-itp/data-analyses/blob/main/_shared_utils/shared_utils/geography_utils.py#L170-L192) that does it. Show the steps within the function (the long way), and also create the `geometry` column using `shared_utils`.

In [12]:
#showing steps
stops2 = stops.assign(
      geometry = gpd.points_from_xy(stops['stop_lon'], stops['stop_lat'], 
      crs = 'WGS84'))

gdf = gpd.GeoDataFrame(stops2)

In [13]:
gdf.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,operator,agency,geometry
396506,246,22nd_street,37.756972,-122.392492,22nd Street Station,Caltrain,Caltrain,POINT (-122.39249 37.75697)
396507,246,22nd_street,37.756972,-122.392492,22nd Street,Caltrain,Caltrain,POINT (-122.39249 37.75697)


In [14]:
gdf.shape

(2017, 8)

#### <b> QUESTION </b> Trying to use the function but I can't seem to find it in geography utils. Some of the other functions are showing though, just not the function "create point geometry."

In [15]:
dir(shared_utils.geography_utils)

['CA_NAD83Albers',
 'CA_StatePlane',
 'Pipeable',
 'SQ_MI_PER_SQ_M',
 'WGS84',
 '_',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'add_count',
 'aggregate_by_geography',
 'anti_join',
 'arrange',
 'attach_geometry',
 'calitp',
 'case_when',
 'collect',
 'complete',
 'count',
 'create_point_geometry',
 'distinct',
 'expand',
 'extract',
 'filter',
 'full_join',
 'gather',
 'gpd',
 'group_by',
 'head',
 'if_else',
 'inner_join',
 'join',
 'left_join',
 'make_routes_shapefile',
 'mutate',
 'nest',
 'os',
 'pd',
 'pipe',
 'rename',
 'right_join',
 'select',
 'semi_join',
 'separate',
 'shapely',
 'show_query',
 'spread',
 'summarize',
 'tbl',
 'top_n',
 'transmute',
 'ungroup',
 'unite',
 'unnest']

## Spatial Join (which points fall into which polygon)

This URL gives you CA county boundaries: https://gis.data.ca.gov/datasets/CALFIRE-Forestry::california-county-boundaries/explore?location=37.246136%2C-119.002032%2C6.12

* Go to "I want to use this" > View API Resources > copy link for geojson 
* [Link to Geojson](https://opendata.arcgis.com/datasets/8713ced9b78a4abb97dc130a691a8695_0.geojson)
* Read in the geojson with `geopandas` and make it a geodataframe: `gpd.read_file(LONG_URL_PATH)`
* Double check that the coordinate reference system is the same for both gdfs using `gdf.crs`. If not, change it so they are the same.
* Spatial join stops to counties: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html)
    * Play with inner join or left join, what's the difference? Which one do you want?
    * Play with switching around the left_df and right_df, what's the right order?


In [16]:
geojson = gpd.read_file('https://opendata.arcgis.com/datasets/8713ced9b78a4abb97dc130a691a8695_0.geojson')

#### Checking to see if the 2 dataframes have the same coordinate system, WGS 84 

In [17]:
gdf.crs 

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [18]:
geojson.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

#### Spatial Join

* Left join: with Cal ITP data on the left is the same as an inner join between Cal ITP & county boundaries data frame. 

* The left join with county data on the left was totally crazy? There were many more rows. I guess some rows from the Cal ITP data didn't match up with any of the counties.

In [19]:
#inner
spatial_inner_data = gpd.sjoin(gdf, geojson, how='inner')

spatial_inner_data.shape

(2017, 19)

In [20]:
#left join 1
spatial_left1 = gpd.sjoin(gdf, geojson, how='left')

spatial_left1.shape

(2017, 19)

In [21]:
#checking to see if inner and first left join data frame are the same..false. I checked in the spreadsheets and realized the
#rows were all jumbled 
spatial_left1.equals(spatial_inner_data)

False

In [22]:
#left join 2 switching data frames
spatial_left2 = gpd.sjoin(geojson,gdf, how='left')


In [23]:
#this has way more columns..looks totally inaccurate
spatial_left2.shape

(2079, 19)

#### QUESTION Should I drop duplicates? I noticed for both the inner & left data from with Cal ITP on the left, the rows went from 1363 to 867.

In [24]:
spatial_inner_data = spatial_inner_data.drop_duplicates()

In [25]:
spatial_inner_data.shape

(949, 19)

## By county: count number of stops and stops per sq_mi
* By county: count number of stops and stops per sq_mi.
    * Hint 1: Start with a CRS with units in feet or meters, then do a conversion to sq mi. [CRS in shared_utils](https://github.com/cal-itp/data-analyses/blob/main/_shared_utils/shared_utils/geography_utils.py)
    * Hint 2: to find area, you can create a new column and calculate `gdf.geometry.area`. [geometry manipulations docs](https://geopandas.org/en/stable/docs/user_guide/geometric_manipulations.html)
    
<b>  Notes from 1/7/2021 </b>
* Several California CRS some are in feet some are in meter
* Lat and lon will be stored as Wgs84, units are decimal meters
* Most of the time you have to convert
* Want main data frame on the left, but when you map it have to replace geometry column of bus points to the county geometry
* Left/Inner and aggregate on county level, count how many stops, merge in shape file
* By county: count number of stops and stops per sq_mi.
    * Hint 1: Start with a CRS with units in feet or meters, then do a conversion to sq mi. [CRS in shared_utils](https://github.com/cal-itp/data-analyses/blob/main/_shared_utils/shared_utils/geography_utils.py)
    * Hint 2: to find area, you can create a new column and calculate `gdf.geometry.area`. [geometry manipulations docs](https://geopandas.org/en/stable/docs/user_guide/geometric_manipulations.html)
   

#### Stops per County first

In [26]:
#changing data type to 2229 
spatial_inner_data = spatial_inner_data.to_crs(epsg =2229)

In [27]:
#subsetting for the columns of interest.
subset = spatial_inner_data[['geometry','COUNTY_NAME','stop_id']]

In [28]:
subset.head(2)

,geometry,COUNTY_NAME,stop_id
396506,POINT (5290484.166 3218221.779),San Francisco,22nd_street
396507,POINT (5290484.166 3218221.779),San Francisco,22nd_street


In [29]:
#aggregating stops by counting unique stops per county.
bus_stops_county = subset.dissolve(by='COUNTY_NAME', aggfunc ='nunique').reset_index()

In [30]:
#have to rename bc I keep getting an error when merging
bus_stops_county = bus_stops_county.rename(columns={'index_left': 'left', 'index_right':'right'})

In [31]:
#converting geojson to correct crs of bus stops.
geojson_epsg229 = geojson.to_crs(bus_stops_county.crs)

In [32]:
#merge in geojson with stops by county..
bus_stops_joined = gpd.sjoin(geojson_epsg229,bus_stops_county,  how='inner')

In [33]:
#drop columns I don't need
bus_stops_joined = bus_stops_joined[['geometry','COUNTY_NAME_left','stop_id']]

In [34]:
#rename columns
bus_stops_joined = bus_stops_joined.rename(columns = {'stop_id': 'Count_of_Stops', 'COUNTY_NAME_left': 'County_Name'})

In [35]:
#looking at dataframe
bus_stops_joined 

,geometry,County_Name,Count_of_Stops
0,"MULTIPOLYGON (((5327843.636 3270649.517, 53281...",Alameda,77
6,"MULTIPOLYGON (((5531603.152 3330218.847, 55316...",Contra Costa,29
23,"MULTIPOLYGON (((5874128.469 3143301.015, 58787...",Merced,480
37,"MULTIPOLYGON (((5297883.323 3264134.097, 53070...",San Francisco,60
40,"MULTIPOLYGON (((5289433.313 3200486.689, 52944...",San Mateo,57
42,"MULTIPOLYGON (((5547146.542 3107821.318, 55532...",Santa Clara,61
49,"MULTIPOLYGON (((5737859.277 3295948.570, 57422...",Stanislaus,5


In [36]:
#just making sure my results above are  right, I am surprised to see Merced County has the most stops
testing = spatial_inner_data.groupby(['COUNTY_NAME']).agg({'stop_id':'nunique'}).reset_index()
testing

,COUNTY_NAME,stop_id
0,Alameda,77
1,Contra Costa,29
2,Merced,480
3,San Francisco,60
4,San Mateo,57
5,Santa Clara,61
6,Stanislaus,5


#### Stops per Square Mile - why did my county names turn into "1"?

In [37]:
#square miles conversion
sq_mi_per_sq_m = 3.86 * 10**-7

In [38]:
bus_stops_sqmile = gpd.sjoin(geojson_epsg229,subset,  how='left')

In [39]:
#Calculate pe square mile
bus_stops_sqmile['Square_Miles'] = bus_stops_sqmile['geometry'].area * sq_mi_per_sq_m

In [40]:
#drop columns I don't need
bus_stops_sqmile = bus_stops_sqmile[['geometry','COUNTY_NAME_left','stop_id','Square_Miles']]

In [ ]:
#Aggregate
bus_stops_sqmile = bus_stops_sqmile.dissolve(by='Square_Miles', aggfunc ='nunique').reset_index()

In [ ]:
bus_stops_sqmile["Stops_per_Sq"] = bus_stops_sqmile['stop_id'] / bus_stops_sqmile['Square_Miles']

In [ ]:
bus_stops_sqmile